# Exergy Analysis (Gouy-Stodola Theorem)

Quantifies the *quality* of energy lost to pipe friction, not just the head loss itself. Frictional irreversibility generates entropy; the Gouy-Stodola theorem converts that entropy generation into a destroyed-exergy (lost work potential) figure in Watts.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd

from src.simulation.scenario import run_simulation, PipeScenario
from src.simulation.sensitivity import sweep_parameter

## Single-scenario exergy breakdown

In [ ]:
result = run_simulation(
    diameter_m=0.0508,      # 2 inch
    flow_rate_m3s=0.0005,
    length_m=100.0,
    fittings={'elbow_90_standard': 4, 'gate_valve_open': 1},
)

print(f"Hydraulic power:      {result.pump.hydraulic_power_W:8.2f} W")
print(f"Pump shaft power:     {result.pump.shaft_power_W:8.2f} W")
print(f"Exergy destroyed:     {result.exergy.exergy_destruction_W:8.2f} W")
print(f"  as % of shaft power: {result.exergy.exergy_destruction_fraction:6.1%}")
print(f"Entropy generation:   {result.exergy.entropy_generation_rate_W_per_K:.4f} W/K")

## How does exergy destruction scale with diameter?

Smaller pipes → higher velocity → higher friction losses → more exergy destroyed for the same delivered flow.

In [ ]:
base = PipeScenario(diameter_m=0.0127, flow_rate_m3s=0.0005, length_m=100.0,
                    fittings={'elbow_90_standard': 4, 'gate_valve_open': 1})
diameters = np.linspace(0.0127, 0.1016, 12)
df = sweep_parameter(base, 'diameter_m', diameters)
df[['diameter_m', 'total_loss_m', 'exergy_destroyed_W']]

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(x=df['diameter_m'], y=df['exergy_destroyed_W'],
                          mode='lines+markers', name='Exergy Destroyed [W]'))
fig.update_layout(title='Exergy Destroyed vs. Pipe Diameter',
                   xaxis_title='Diameter (m)', yaxis_title='Exergy Destroyed (W)',
                   template='plotly_white')
fig.show()

## Energy flow Sankey for the worst case (smallest diameter)

In [ ]:
from src.plots.sankey import energy_flow_sankey

worst_case = run_simulation(diameter_m=0.0127, flow_rate_m3s=0.0005, length_m=100.0,
                             fittings={'elbow_90_standard': 4, 'gate_valve_open': 1},
                             label='1/2 inch (worst case)')
energy_flow_sankey(worst_case).show()